#### Import libraries

In [1]:
import geopandas as gpd
from sqlalchemy import text, create_engine
import numpy as np
import pandas as pd
from matplotlib.colors import to_hex, to_rgb
from matplotlib.patches import Patch
import seaborn as sns
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import PercentFormatter
import calendar

#### Connection DB

In [2]:
# Connection with the DB
engine = create_engine("postgresql://ual:@cirrus.ita.chalmers.se/ual_prototyping",
                       connect_args={"client_encoding": "utf8"})
conn = engine.connect()

#### Import data

In [3]:
YEARS = (2023,
         2024,
         2025)

PARK_SCHEMA = "processed"
PARK_TABLE_PREFIX = "swe_national_parks_movement_point_types"
PARK_COL = "namn"
PARKS = (
    "Abisko",
    "Skuleskogen",
)
import parking table
PARKING_TABLE_PREFIX = boudaries.swe_national_parks_parking

STATS_SCHEMA = "processed"
STATS_TABLE = "stats_devices"
STATS_YEAR_COL = "year"
STATS_MONTH_COL = "month"
STATS_DEVICE_COL = "devices"

TIMESTAMP_COL = "arrive_time"
DEVICE_COL = "device_aid"
GEOM_COL = "geom"
EPSG = 4326

default_palette = {
    "Abisko": {
        2023: "#A9D6E5",
        2024: "#61A5C2",
        2025: "#277DA1",
    },

    # Woodland greens
    "Skuleskogen": {
        2023: "#B7D69A",
        2024: "#7AAA55",
        2025: "#4F772D",
    },
}

park_labels = {
        "Abisko": "Abisko",
        "Skuleskogen": "Skuleskogen",
}

In [4]:
# Import aggregated national park data
def read_spatial_table(
        engine,
        schema,
        table,
        *,
        geom_col=GEOM_COL,
        epsg=EPSG,
):
    query = text(
        f'SELECT * FROM "{schema}"."{table}"'
    )

    gdf = gpd.read_postgis(
        query,
        con=engine,
        geom_col=geom_col,
    )

    if gdf.crs is None:
        gdf = gdf.set_crs(
            epsg=epsg,
            allow_override=True,
        )
    elif gdf.crs.to_epsg() != epsg:
        gdf = gdf.to_crs(
            epsg=epsg,
        )

    return gdf


def import_park_data(
        engine,
        *,
        years=YEARS,
        schema=PARK_SCHEMA,
        table_prefix=PARK_TABLE_PREFIX,
        geom_col=GEOM_COL,
):
    output = {}

    for year in years:
        table = f"{table_prefix}_{year}"

        output[year] = read_spatial_table(
            engine=engine,
            schema=schema,
            table=table,
            geom_col=geom_col,
        )

        print(
            f"{year}: "
            f"{len(output[year]):,} rows"
        )

    return output


def read_table(
        engine,
        schema,
        table,
):
    query = text(
        f'SELECT * FROM "{schema}"."{table}"'
    )

    return pd.read_sql(
        query,
        con=engine,
    )


# Import national-park movement tables by year

park_data = import_park_data(
    engine=engine,
)

parks_2023 = park_data[2023]
parks_2024 = park_data[2024]
parks_2025 = park_data[2025]

# Import device statistics

stats_device = read_table(
    engine=engine,
    schema=STATS_SCHEMA,
    table=STATS_TABLE,
)

print(
    f"{STATS_SCHEMA}.{STATS_TABLE}: "
    f"{len(stats_device):,} rows"
)

2023: 3,744,646 rows
2024: 4,129,285 rows
2025: 4,757,239 rows
processed.stats_devices: 36 rows


In [7]:
# Create normalized device table
def create_normalized_devices(
        parks_table,
        stats_device,
        *,
        stats_year_col=STATS_YEAR_COL,
        stats_month_col=STATS_MONTH_COL,
        stats_device_col=STATS_DEVICE_COL,
):
    stats = stats_device.copy()

    required_columns = {
        stats_year_col,
        stats_month_col,
        stats_device_col,
    }

    missing_columns = required_columns.difference(
        stats.columns
    )

    if missing_columns:
        raise KeyError(
            "Missing columns in stats_device: "
            f"{sorted(missing_columns)}"
        )

    stats[stats_year_col] = pd.to_numeric(
        stats[stats_year_col],
        errors="coerce",
    )

    stats[stats_month_col] = pd.to_numeric(
        stats[stats_month_col],
        errors="coerce",
    )

    stats[stats_device_col] = pd.to_numeric(
        stats[stats_device_col],
        errors="coerce",
    )

    stats["month"] = pd.to_datetime(
        {
            "year": stats[stats_year_col],
            "month": stats[stats_month_col],
            "day": 1,
        },
        errors="coerce",
    )

    monthly_totals = (
        stats
        .dropna(
            subset=[
                "month",
                stats_device_col,
            ]
        )
        .groupby(
            "month",
            as_index=False,
        )
        .agg(
            total_devices=(
                stats_device_col,
                "sum",
            )
        )
    )

    normalized_devices = (
        parks_table
        .merge(
            monthly_totals,
            on="month",
            how="left",
            validate="many_to_one",
        )
    )

    normalized_devices["normalized_devices"] = (
        normalized_devices["unique_devices"]
        .div(
            normalized_devices["total_devices"]
            .replace(0, pd.NA)
        )
    )

    normalized_devices[
        "normalized_devices_percent"
    ] = (
            normalized_devices["normalized_devices"]
            * 100
    )

    return normalized_devices[
        [
            "month",
            "park",
            "total_records",
            "unique_devices",
            "total_devices",
            "normalized_devices",
            "normalized_devices_percent",
        ]
    ]


normalized_devices = create_normalized_devices(
    parks_table=parks_table,
    stats_device=stats_device,
)

normalized_devices.head()

NameError: name 'parks_table' is not defined

In [ ]:
def get_monthly_device_distribution(
        stats_device,
        *,
        years=YEARS,
        year_col=STATS_YEAR_COL,
        month_col=STATS_MONTH_COL,
        device_col=STATS_DEVICE_COL,
):
    """
    Calculate the monthly distribution of devices by year.

    The monthly share represents the percentage of the annual
    device total recorded in each month.
    """
    required_columns = [
        year_col,
        month_col,
        device_col,
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in stats_device.columns
    ]

    if missing_columns:
        raise KeyError(
            f"Missing required columns: {missing_columns}"
        )

    stats_table = stats_device[
        required_columns
    ].copy()

    stats_table[year_col] = pd.to_numeric(
        stats_table[year_col],
        errors="coerce",
    )

    stats_table[month_col] = pd.to_numeric(
        stats_table[month_col],
        errors="coerce",
    )

    stats_table[device_col] = pd.to_numeric(
        stats_table[device_col],
        errors="coerce",
    )

    stats_table = stats_table.dropna(
        subset=[
            year_col,
            month_col,
            device_col,
        ]
    )

    stats_table = stats_table[
        stats_table[year_col].isin(years)
        & stats_table[month_col].between(1, 12)
        ].copy()

    # Combine rows if the table contains more than one
    # observation for the same year and month.
    stats_table = (
        stats_table
        .groupby(
            [
                year_col,
                month_col,
            ],
            as_index=False,
        )
        .agg(
            devices=(
                device_col,
                "sum",
            )
        )
    )

    complete_months = pd.MultiIndex.from_product(
        [
            years,
            range(1, 13),
        ],
        names=[
            year_col,
            month_col,
        ],
    ).to_frame(
        index=False
    )

    stats_table = complete_months.merge(
        stats_table,
        how="left",
        on=[
            year_col,
            month_col,
        ],
    )

    stats_table["devices"] = (
        stats_table["devices"]
        .fillna(0)
        .astype("int64")
    )

    stats_table["month_name"] = (
        stats_table[month_col]
        .map(calendar.month_abbr.__getitem__)
    )

    annual_devices = (
        stats_table
        .groupby(year_col)["devices"]
        .transform("sum")
    )

    stats_table["annual_devices"] = annual_devices

    stats_table["distribution_pct"] = np.where(
        annual_devices > 0,
        (
                stats_table["devices"]
                / annual_devices
                * 100
        ),
        0,
    )

    stats_table["distribution_pct"] = (
        stats_table["distribution_pct"]
        .round(2)
    )

    stats_table = stats_table[
        [
            year_col,
            month_col,
            "month_name",
            "devices",
            "annual_devices",
            "distribution_pct",
        ]
    ].sort_values(
        [
            year_col,
            month_col,
        ]
    ).reset_index(
        drop=True
    )

    return stats_table


stats_table = get_monthly_device_distribution(
    stats_device=stats_device,
)

display(stats_table)

In [ ]:
def plot_monthly_device_distribution(
        stats_table,
        *,
        year_col=STATS_YEAR_COL,
        month_col=STATS_MONTH_COL,
        value_col="distribution_pct",
        save_path=None,
):
    """Plot the monthly percentage distribution of devices."""
    plt.close("all")
    plt.style.use("default")
    sns.set_style("white")

    year_colors = {
        2023: "#A9D6E5",
        2024: "#61A5C2",
        2025: "#277DA1",
    }

    years = sorted(
        stats_table[year_col].unique()
    )

    palette = {
        year: year_colors.get(
            year,
            "#61A5C2",
        )
        for year in years
    }

    fig, ax = plt.subplots(
        figsize=(14, 7),
    )

    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    sns.lineplot(
        data=stats_table,
        x=month_col,
        y=value_col,
        hue=year_col,
        palette=palette,
        marker="o",
        linewidth=2.2,
        markersize=7,
        ax=ax,
    )

    ax.set_title(
        "Monthly Distribution of Devices",
        fontsize=16,
        fontweight="bold",
        pad=18,
    )

    ax.set_xlabel(
        "Month",
        fontsize=11,
    )

    ax.set_ylabel(
        "Share of Annual Devices (%)",
        fontsize=11,
    )

    ax.set_xticks(
        range(1, 13)
    )

    ax.set_xticklabels(
        [
            calendar.month_abbr[month]
            for month in range(1, 13)
        ]
    )

    ax.grid(
        axis="y",
        color="#D9D9D9",
        linewidth=0.8,
        alpha=0.7,
    )

    ax.grid(
        axis="x",
        visible=False,
    )

    sns.despine(
        ax=ax,
        left=True,
        bottom=False,
    )

    ax.legend(
        title="Year",
        frameon=False,
    )

    plt.tight_layout()

    if save_path:
        fig.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight",
            facecolor="white",
        )

    plt.show()

    return fig, ax


fig, ax = plot_monthly_device_distribution(
    stats_table=stats_table,
)

In [ ]:
# Plot
def plot_combined_devices_by_month(
        data,
        plot_title,
        *,
        years=YEARS,
        parks=PARKS,
        month_col="month",
        park_col="park",
        devices_col="unique_devices",
        normalized_col="normalized_devices_percent",
        save_path=None,
):

    plt.close("all")
    plt.style.use("default")

    sns.set_theme(
        context="notebook",
        style="white",
        font_scale=1.0,
        rc={
            "figure.facecolor": "white",
            "axes.facecolor": "#FCFCFA",
            "axes.edgecolor": "#CBD1C8",
            "axes.labelcolor": "#354239",
            "text.color": "#26382D",
            "xtick.color": "#4D5A51",
            "ytick.color": "#4D5A51",
        },
    )

    month_labels = (
        "Jan",
        "Feb",
        "Mar",
        "Apr",
        "May",
        "Jun",
        "Jul",
        "Aug",
        "Sep",
        "Oct",
        "Nov",
        "Dec",
    )

    required_columns = {
        month_col,
        park_col,
        devices_col,
        normalized_col,
    }

    missing_columns = required_columns.difference(
        data.columns
    )

    if missing_columns:
        raise KeyError(
            "Missing visualization columns: "
            f"{sorted(missing_columns)}"
        )

    years = tuple(years)
    parks = tuple(parks)

    if not years:
        raise ValueError(
            "'years' must contain at least one year."
        )

    if not parks:
        raise ValueError(
            "'parks' must contain at least one park."
        )

    plot_data = data.copy()

    plot_data[month_col] = pd.to_datetime(
        plot_data[month_col],
        errors="coerce",
    )

    plot_data[devices_col] = pd.to_numeric(
        plot_data[devices_col],
        errors="coerce",
    )

    plot_data[normalized_col] = pd.to_numeric(
        plot_data[normalized_col],
        errors="coerce",
    )

    plot_data["year"] = (
        plot_data[month_col].dt.year
    )

    plot_data["calendar_month"] = (
        plot_data[month_col].dt.month
    )

    plot_data = (
        plot_data.loc[
            plot_data[park_col].isin(parks)
            & plot_data["year"].isin(years)
            & plot_data["calendar_month"].notna()
            ]
        .sort_values(
            [
                park_col,
                "year",
                "calendar_month",
            ]
        )
        .copy()
    )

    if plot_data.empty:
        raise ValueError(
            "No observations match the selected parks and years."
        )

    duplicate_rows = plot_data.duplicated(
        subset=[
            park_col,
            "year",
            "calendar_month",
        ],
        keep=False,
    )

    if duplicate_rows.any():
        duplicate_keys = (
            plot_data.loc[
                duplicate_rows,
                [
                    park_col,
                    "year",
                    "calendar_month",
                ],
            ]
            .drop_duplicates()
            .head(10)
            .to_dict("records")
        )

        raise ValueError(
            "Multiple rows were found for the same park, year, "
            "and month. Aggregate the data before plotting. "
            f"Examples: {duplicate_keys}"
        )

    figure_height = max(
        10.5,
        3.25 * len(parks),
    )

    fig, axes = plt.subplots(
        nrows=len(parks),
        ncols=1,
        figsize=(12, figure_height),
        sharex=True,
        squeeze=False,
        constrained_layout=False,
    )

    axes = axes[:, 0]

    x_months = np.arange(
        1,
        13,
        dtype=float,
    )

    group_width = 0.72
    bar_width = group_width / len(years)

    year_offsets = {
        year: (
                      index - (len(years) - 1) / 2
              ) * bar_width
        for index, year in enumerate(years)
    }

    for row, park in enumerate(parks):
        ax_devices = axes[row]
        ax_share = ax_devices.twinx()

        park_palette = park_palettes.get(
            park,
            default_palette,
        )

        park_color = park_palette.get(
            max(years),
            "#405743",
        )

        for year in years:
            color = park_palette.get(
                year,
                park_color,
            )

            yearly_data = (
                plot_data.loc[
                    (plot_data[park_col] == park)
                    & (plot_data["year"] == year),
                    [
                        "calendar_month",
                        devices_col,
                        normalized_col,
                    ],
                ]
                .set_index(
                    "calendar_month"
                )
                .reindex(
                    range(1, 13)
                )
            )

            x_positions = (
                    x_months
                    + year_offsets[year]
            )

            ax_devices.bar(
                x_positions,
                yearly_data[devices_col],
                width=bar_width * 0.88,
                color=color,
                alpha=0.82,
                edgecolor="white",
                linewidth=0.7,
                zorder=2,
            )

            ax_share.plot(
                x_months,
                yearly_data[normalized_col],
                color=color,
                linewidth=2.2,
                marker="o",
                markersize=4.8,
                markerfacecolor=color,
                markeredgecolor="white",
                markeredgewidth=0.8,
                solid_capstyle="round",
                zorder=4,
            )

        ax_devices.set_title(
            park_labels.get(
                park,
                park,
            ),
            loc="center",
            fontsize=14,
            fontweight="bold",
            color=park_color,
            pad=14,
        )

        ax_devices.set_ylabel(
            "Unique devices",
            fontsize=10.5,
            fontweight="medium",
            labelpad=11,
            color="#354239",
        )

        ax_share.set_ylabel(
            "Share of monthly devices",
            fontsize=10.5,
            fontweight="medium",
            labelpad=13,
            color=park_color,
        )

        ax_devices.yaxis.set_major_locator(
            MaxNLocator(
                nbins=5,
                integer=True,
                min_n_ticks=3,
            )
        )

        ax_devices.yaxis.set_major_formatter(
            StrMethodFormatter(
                "{x:,.0f}"
            )
        )

        ax_share.yaxis.set_major_locator(
            MaxNLocator(
                nbins=5,
                min_n_ticks=3,
            )
        )

        ax_share.yaxis.set_major_formatter(
            PercentFormatter(
                xmax=100,
                decimals=2,
            )
        )

        ax_devices.grid(
            axis="y",
            color="#DDE2DA",
            linewidth=0.8,
            alpha=0.75,
            zorder=0,
        )

        ax_devices.grid(
            axis="x",
            visible=False,
        )

        ax_share.grid(False)
        ax_devices.set_axisbelow(True)

        ax_devices.tick_params(
            axis="both",
            labelsize=9,
            length=0,
            pad=6,
        )

        ax_share.tick_params(
            axis="y",
            labelsize=9,
            length=0,
            pad=7,
            colors=park_color,
        )

        ax_devices.spines["top"].set_visible(False)
        ax_devices.spines["right"].set_visible(False)

        ax_devices.spines["left"].set_color(
            "#CBD1C8"
        )

        ax_devices.spines["bottom"].set_color(
            "#CBD1C8"
        )

        ax_share.spines["top"].set_visible(False)
        ax_share.spines["left"].set_visible(False)
        ax_share.spines["bottom"].set_visible(False)

        ax_share.spines["right"].set_color(
            park_color
        )

        ax_share.spines["right"].set_linewidth(
            0.9
        )

        ax_devices.margins(
            y=0.12,
        )

        ax_share.margins(
            y=0.15,
        )

    axes[-1].set_xlim(
        0.45,
        12.55,
    )

    axes[-1].set_xticks(
        x_months,
    )

    axes[-1].set_xticklabels(
        month_labels,
        fontsize=9.5,
    )

    axes[-1].set_xlabel(
        "Month",
        fontsize=11,
        fontweight="medium",
        labelpad=14,
    )

    year_legend_colors = (
        "#B8CFB1",
        "#739B6B",
        "#35583A",
    )

    year_handles = [
        Patch(
            facecolor=(
                year_legend_colors[index]
                if index < len(year_legend_colors)
                else default_palette.get(
                    year,
                    "#758C70",
                )
            ),
            edgecolor="none",
            label=str(year),
        )
        for index, year in enumerate(years)
    ]

    metric_handles = [
        Patch(
            facecolor="#758C70",
            edgecolor="white",
            linewidth=0.7,
            alpha=0.82,
            label="Unique devices",
        ),
        Line2D(
            [0],
            [0],
            color="#405743",
            marker="o",
            markerfacecolor="#405743",
            markeredgecolor="white",
            markeredgewidth=0.8,
            linewidth=2.2,
            label="Share of all devices in Sweden",
        ),
    ]

    fig.suptitle(
        plot_title,
        fontsize=19,
        fontweight="bold",
        color="#26382D",
        x=0.5,
        y=0.982,
        ha="center",
    )

    fig.text(
        0.5,
        0.953,
        (
            "Bars show monthly unique devices; lines show each "
            "park’s share of all devices observed in Sweden."
        ),
        ha="center",
        va="top",
        fontsize=10.5,
        color="#667168",
    )

    year_legend = fig.legend(
        handles=year_handles,
        title="Year",
        loc="upper right",
        bbox_to_anchor=(
            0.965,
            0.955,
        ),
        frameon=False,
        ncol=len(years),
        columnspacing=1.5,
        handlelength=1.7,
        handletextpad=0.5,
        borderaxespad=0,
        fontsize=9.5,
        title_fontsize=10,
    )

    fig.add_artist(
        year_legend
    )

    fig.legend(
        handles=metric_handles,
        loc="lower center",
        bbox_to_anchor=(
            0.5,
            0.045,
        ),
        frameon=False,
        ncol=2,
        columnspacing=2.5,
        handlelength=2.4,
        handletextpad=0.7,
        borderaxespad=0,
        fontsize=10,
    )

    fig.subplots_adjust(
        left=0.10,
        right=0.87,
        top=0.875,
        bottom=0.125,
        hspace=0.44,
    )

    if save_path:
        output_directory = os.path.dirname(
            save_path
        )

        if output_directory:
            os.makedirs(
                output_directory,
                exist_ok=True,
            )

        fig.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight",
            pad_inches=0.28,
            facecolor="white",
            edgecolor="none",
        )

    plt.show()

In [ ]:
plot_combined_devices_by_month(
    data=normalized_devices,
    plot_title=(
        "How Visits to Sweden’s National Parks "
        "Change Throughout the Years"
    )
)

In [ ]:
normalized_devices